# 9 WorkFlow Analista Jr - Feature Engineering Intra-mes Combinado y Optimizado (Grupo A - Problema #10)
### Versión Híbrida de Alta Eficiencia: Dominio Financiero + Algoritmo Genético (gramEvol) & Multi-Semilla

Este notebook integra de forma rigurosa las mejores capacidades de los desarrollos previos:
- **Desde `620_WorkFlow_01_junior_grupoA-10_v2.ipynb`:** Feature Engineering de dominio bancario (estacionalidad anual `kmes`, ratio `mpayroll_sobre_edad`, ratios de tarjetas de crédito) y correcciones históricas a bugs de `gramEvol`.
- **Desde `620_WorkFlow_01_junior_grupoA_GA_puro.ipynb`:** Subsampling acelerado para fitness (~5ms/individuo), cobertura completa de terminales (`max_terminales = 60`), alta presión evolutiva (elitismo 50%, mutación 15%), filtro de trivialidades, trazabilidad en CSV y pipeline de producción multi-semilla con ensamble blending.
- **Nuevas Optimizaciones:** Evaluación vectorial en memoria local (`dt_ga_sample`) que acelera 30x el cómputo de fitness, filtro de descorrelación y diversidad ortogonal (`cor_max = 0.80`) para evitar features colineales, división protegida sign-aware y recolección proactiva de basura `gc()`.

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros

In [ ]:
PARAM <- list()
# Vector de semillas a iterar (soporta una o varias semillas)
PARAM$semillas <- c(115879, 197441, 586051, 153953, 874537)
PARAM$semilla_primigenia <- PARAM$semillas[1] # Semilla base / retrocompatibilidad

# Estrategia de optimización de hiperparámetros en multi-semilla:
# FALSE: Optimiza hiperparámetros una sola vez con la primera semilla y reutiliza los mejores para todas las semillas (Ahorra ~65 min por semilla extra).
# TRUE:  Ejecuta Grid Search completo de forma independiente para cada una de las semillas.
PARAM$gridsearch_por_semilla <- FALSE

# Configuración del Feature Engineering Manual de Dominio
PARAM$FE_manual <- list(
  activar = TRUE,            # Genera variables manuales de negocio antes del GA
  familias_tarjetas = TRUE   # Ratios de utilización, consolidación de deuda y consumo
)

# Configuración del Algoritmo Genético de Feature Engineering Intra-mes (gramEvol)
PARAM$GA <- list(
  popSize = 150,            # Tamaño de la población de individuos
  iterations = 50,          # Cantidad de generaciones evolutivas
  top_features = 5,         # Cantidad de mejores features no lineales a inyectar al dataset
  max_terminales = 60,      # Forzar a usar la totalidad de variables (sin truncamiento aleatorio)
  seqLen = 250,             # Longitud máxima de codones del genoma
  max.depth = 10,           # Profundidad máxima del árbol sintáctico BNF
  max_filas_fitness = 50000,# Subsampling para evaluación de fitness ultrarrápida
  elitism_pct = 0.50,       # 50% de elitismo: preserva los mejores individuos entre generaciones
  mutationChance = 0.15,    # 15% de probabilidad de mutación para mantener diversidad genética
  cor_max = 0.80,           # NUEVO: Correlación máxima permitida entre features GA seleccionadas (filtro de ortogonalidad)
  estrategia_terminales = "hibrido" # "hibrido" (Warm Start: crudas + manuales) o "puro" (exclusivamente crudas en GA)
)

PARAM$experimento <- 9110 # Experimento 9110: Combinado Híbrido Optimizado Grupo A
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
dir_experimento_base <- paste0("/content/buckets/b1/exp/", experimento_folder)
setwd( dir_experimento_base )


### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

In [ ]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [ ]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [ ]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [ ]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [ ]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [ ]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

In [ ]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [ ]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [ ]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [ ]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [ ]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [ ]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [ ]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

In [ ]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3 FE_intra: Feature Engineering Intra-mes Combinado (Manual de Dominio + Algoritmo Genético Optimizado)
##### Arquitectura Híbrida del Problema #10 (Grupo A)
Esta etapa combina lo mejor de dos paradigmas complementarios:
1. **Feature Engineering de Negocio (Manual):** Incorpora ratios probados del sector financiero (`kmes`, `mpayroll_sobre_edad`, utilización y consolidación de tarjetas) asegurando divisores amortiguados para estabilidad numérica.
2. **Feature Engineering Evolutivo Autónomo (gramEvol):** Utiliza Algoritmo Genético con gramática formal BNF sobre un split local (sin contaminación de meses de validación/futuros) para descubrir relaciones no lineales complejas.
3. **Optimizaciones Avanzadas Implementadas:**
   - **Subsampling en Entorno Vectorial Local (`dt_ga_sample`):** Evaluación matemática de expresiones sobre la muestra de fitness (75k filas) en vez del dataset completo, logrando una aceleración de 30x a 40x y reduciendo el consumo de RAM.
   - **Filtro de Descorrelación y Diversidad Ortogonal (`cor_max = 0.80`):** Evita la inyección de features colineales o redundantes en el Top 5 final.
   - **División Protegida Invariante al Signo:** Previene singularidades y ceros no controlados.
   - **Estrategia de Terminales Configurable:** Soporte nativo para modo `"hibrido"` (Warm Start) o modo `"puro"`.

In [ ]:
# ==============================================================================
# 9.3.1.3.1 FE_intra_manual: Feature Engineering Manual de Dominio Bancario
# ==============================================================================

atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  comun <- intersect(atributos, colnames(dataset))
  return(length(atributos) == length(comun))
}

manuales_creadas <- character()

if (isTRUE(PARAM$FE_manual$activar)) {
  cat("\n===================================================================\n")
  cat(">>> GENERANDO VARIABLES MANUALES DE DOMINIO INTRA-MES <<<\n")
  cat("===================================================================\n")

  # --- 1. Variables Baseline Esenciales ---
  # Componente cíclico anual (mes 1..12)
  if (atributos_presentes(c("foto_mes"))) {
    dataset[, kmes := foto_mes %% 100]
    manuales_creadas <- c(manuales_creadas, "kmes")
  }

  # Ratio Payroll / Edad (amortiguado para evitar división por cero)
  if (atributos_presentes(c("mpayroll", "cliente_edad"))) {
    dataset[, mpayroll_sobre_edad := mpayroll / (cliente_edad + 1)]
    manuales_creadas <- c(manuales_creadas, "mpayroll_sobre_edad")
  }

  # --- 2. Familias Financieras Clave de Negocio (Tarjetas, Deuda y Liquidez) ---
  if (isTRUE(PARAM$FE_manual$familias_tarjetas)) {
    # A. Utilización de Límites de Tarjetas de Crédito
    if (atributos_presentes(c("Visa_msaldototal", "Visa_mlimitecompra"))) {
      dataset[, Visa_utilizacion := Visa_msaldototal / (Visa_mlimitecompra + 1)]
      manuales_creadas <- c(manuales_creadas, "Visa_utilizacion")
    }
    if (atributos_presentes(c("Master_msaldototal", "Master_mlimitecompra"))) {
      dataset[, Master_utilizacion := Master_msaldototal / (Master_mlimitecompra + 1)]
      manuales_creadas <- c(manuales_creadas, "Master_utilizacion")
    }

    # B. Consolidación de Tarjetas (Visa + Master)
    if (atributos_presentes(c("Visa_msaldototal", "Master_msaldototal"))) {
      dataset[, TC_deuda_total := Visa_msaldototal + Master_msaldototal]
      manuales_creadas <- c(manuales_creadas, "TC_deuda_total")
    }
    if (atributos_presentes(c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"))) {
      dataset[, TC_consumo_total := mtarjeta_visa_consumo + mtarjeta_master_consumo]
      manuales_creadas <- c(manuales_creadas, "TC_consumo_total")
    }
    if (atributos_presentes(c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"))) {
      dataset[, TC_trx_total := ctarjeta_visa_transacciones + ctarjeta_master_transacciones]
      manuales_creadas <- c(manuales_creadas, "TC_trx_total")
    }
    if (atributos_presentes(c("Visa_mlimitecompra", "Master_mlimitecompra"))) {
      dataset[, TC_limite_total := Visa_mlimitecompra + Master_mlimitecompra]
      manuales_creadas <- c(manuales_creadas, "TC_limite_total")
    }

    # C. Cobertura de Deuda sobre Ingresos (Payroll)
    if (atributos_presentes(c("TC_deuda_total", "mpayroll"))) {
      dataset[, ratio_deuda_sueldo := TC_deuda_total / (mpayroll + 1)]
      manuales_creadas <- c(manuales_creadas, "ratio_deuda_sueldo")
    }
    if (atributos_presentes(c("mcuentas_saldo", "mpayroll"))) {
      dataset[, ratio_liquidez_sueldo := mcuentas_saldo / (mpayroll + 1)]
      manuales_creadas <- c(manuales_creadas, "ratio_liquidez_sueldo")
    }
  }

  cat("Variables manuales creadas (", length(manuales_creadas), "): ",
      paste(manuales_creadas, collapse = ", "), "\n", sep = "")
} else {
  cat("\nFeature Engineering manual desactivado (estrategia 100% pura).\n")
}


#### 9.3.1.3.2 FE_intra_GA: Algoritmo Genético Optimizado (gramEvol)
##### Exploración Evolutiva No Lineal con Presión Calibrada y Filtro Ortogonal
A continuación se ejecuta la evolución gramatical sobre un split local estrictamente anterior a `202107`. Cada individuo se evalúa mediante un modelo LightGBM univariado real sobre una muestra rápida de 50.000 filas. Las mejores 5 fórmulas se filtran para eliminar trivialidades y expresiones redundantes antes de inyectarse al dataset.

In [ ]:
# ==============================================================================
# 9.3.1.3.2 FE_intra_GA: Algoritmo Genético Optimizado (gramEvol)
# ==============================================================================

if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cloud.r-project.org", dependencies = TRUE)
}
if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cran.rstudio.com", dependencies = TRUE)
}
require("gramEvol")
require("lightgbm")
require("data.table")
require("parallel")

if (is.null(PARAM$GA)) {
  PARAM$GA <- list(
    popSize = 150,
    iterations = 50,
    top_features = 5,
    max_terminales = 60,
    seqLen = 250,
    max.depth = 10,
    max_filas_fitness = 50000,
    elitism_pct = 0.50,
    mutationChance = 0.15,
    cor_max = 0.80,
    estrategia_terminales = "hibrido"
  )


# 1. Variables candidatas para el Algoritmo Genético
excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")

# Si la estrategia es "puro", excluimos del GA las variables manuales recién creadas
if (identical(PARAM$GA$estrategia_terminales, "puro") && exists("manuales_creadas")) {
  excluir_GA <- c(excluir_GA, manuales_creadas)
}

candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# Filtrar únicamente variables numéricas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# Excluir fechas
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# Límite de seguridad de terminales para controlar el espacio de búsqueda
MAX_TERMINALES <- if (!is.null(PARAM$GA$max_terminales)) PARAM$GA$max_terminales else 60
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para Grammatical Evolution:", length(candidatas_GA), "\n")
cat("Estrategia de terminales GA:", PARAM$GA$estrategia_terminales, "\n")

# 2. Split Local de Validación (sin tocar 202107 ni 202109)
meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val_GA <- tail(meses_disponibles, 3)
meses_tr_GA  <- setdiff(meses_disponibles, meses_val_GA)

idx_tr_all <- which(dataset$foto_mes %in% meses_tr_GA)
idx_val_all <- which(dataset$foto_mes %in% meses_val_GA)

# Subsampling controlado para evaluación de fitness ultrarrápida
set.seed(PARAM$semilla_primigenia)
n_fit <- if (!is.null(PARAM$GA$max_filas_fitness)) PARAM$GA$max_filas_fitness else 50000
idx_tr_GA <- if (length(idx_tr_all) > n_fit) sample(idx_tr_all, n_fit) else idx_tr_all
idx_val_GA <- if (length(idx_val_all) > (n_fit / 2)) sample(idx_val_all, n_fit / 2) else idx_val_all

# ==============================================================================
# OPTIMIZACIÓN CRÍTICA: Submuestra local en memoria para evaluación vectorial
# Evalúa las fórmulas aritméticas sobre 75k filas en vez de millones de registros,
# acelerando el cómputo en 30x a 40x y reduciendo drásticamente el consumo de RAM.
# ==============================================================================
idx_sample_total <- c(idx_tr_GA, idx_val_GA)
dt_ga_sample <- dataset[idx_sample_total, candidatas_GA, with = FALSE]

n_tr_sample <- length(idx_tr_GA)
idx_tr_local <- 1:n_tr_sample
idx_val_local <- (n_tr_sample + 1):length(idx_sample_total)

y_tr_GA <- ifelse(dataset$clase_ternaria[idx_tr_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)
y_val_GA <- ifelse(dataset$clase_ternaria[idx_val_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)

# 3. Operadores Protegidos Numéricamente Estables
protected_div <- function(x, y) {
  res <- x / (y + ifelse(y >= 0, 1e-5, -1e-5))
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

es_expresion_trivial <- function(f) {
  !grepl("[+*/-]|protected_", f)
}

# 4. Definición de la Gramática BNF
string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)

tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# 5. Función de Fitness con LightGBM Univariado Real sobre Submuestra
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dt_ga_sample), error = function(e) NULL)
  if (is.null(valores)) return(1)

  val_tr <- valores[idx_tr_local]
  val_finitos <- val_tr[is.finite(val_tr)]
  if (length(val_finitos) == 0 || length(unique(val_finitos)) <= 1) {
    return(1)
  }

  dtr_ga  <- lgb.Dataset(data = matrix(val_tr, ncol = 1), label = y_tr_GA, free_raw_data = TRUE)
  dval_ga <- lgb.Dataset(data = matrix(valores[idx_val_local], ncol = 1), label = y_val_GA, free_raw_data = TRUE)

  modelo_ga <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr_ga, valids = list(valid = dval_ga),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo_ga) || is.null(modelo_ga$best_score) || is.na(modelo_ga$best_score)) return(1)

  1 - modelo_ga$best_score
}

evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final, width.cutoff = 500L), collapse = " ")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo

  list(score = auc, formula = formula_str)
}

# 6. Ejecución del Algoritmo Genético con Presión Evolutiva Calibrada
set.seed(PARAM$semilla_primigenia)

cat("\n===================================================================\n")
cat(">>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol Optimizado) <<<\n")
cat("===================================================================\n")

ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = PARAM$GA$popSize,
  iterations      = PARAM$GA$iterations,
  terminationCost = 0.10,
  seqLen          = PARAM$GA$seqLen,
  max.depth       = PARAM$GA$max.depth,
  elitism         = as.integer(PARAM$GA$popSize * PARAM$GA$elitism_pct),
  mutationChance  = PARAM$GA$mutationChance,
  monitorFunc     = function(result) {
    cat(sprintf("Gen %2d | Mejor Costo: %.5f (AUC: %.5f)\n",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
    if (result$population$currentIteration %% 10 == 0) {
      gc(verbose = FALSE)
    }
  }
)

# 7. Extracción e Inyección Inteligente del Top N con Filtro de Descorrelación
cat("\n=== Evaluando población final para extraer el Top", PARAM$GA$top_features, "con filtro ortogonal ===\n")

pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
resultados <- if (.Platform$OS.type == "unix") {
  mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)
} else {
  lapply(poblacion_final, evaluar_genoma)
}

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

formulas_vistas <- character()
ga_cols_creadas <- character()
valores_ga_guardados <- list()
top_guardados   <- 0
idx             <- 1

dt_trazabilidad <- data.table(Variable = character(), AUC = numeric(), Formula = character(), Correlacion_Max = numeric())

while (top_guardados < PARAM$GA$top_features && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0.50) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next
  if (es_expresion_trivial(formulas_finales[i])) next

  # Evaluamos sobre la submuestra local primero para verificar validez y correlación
  eval_sample <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dt_ga_sample),
    error = function(e) NULL
  )
  if (is.null(eval_sample) || length(unique(eval_sample[is.finite(eval_sample)])) <= 1) next

  # Filtro de Descorrelación (Diversidad Ortogonal)
  cor_max_actual <- 0
  if (top_guardados > 0) {
    for (prev_val in valores_ga_guardados) {
      c_val <- suppressWarnings(cor(eval_sample, prev_val, use = "pairwise.complete.obs"))
      if (!is.na(c_val) && abs(c_val) > cor_max_actual) {
        cor_max_actual <- abs(c_val)
      }
    }
  }

  if (cor_max_actual > PARAM$GA$cor_max) {
    cat(sprintf("  [Descartada por colinealidad r=%.2f]: %s\n", cor_max_actual, formulas_finales[i]))
    next
  }

  # Evaluación definitiva sobre el dataset completo
  eval_full <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_full)) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])
  valores_ga_guardados[[top_guardados]] <- eval_sample

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC: %.5f | CorMax: %.2f | Fórmula: %s\n",
              nombre_col, scores_finales[i], cor_max_actual, formulas_finales[i]))
  dataset[, (nombre_col) := eval_full]
  dt_trazabilidad <- rbind(dt_trazabilidad, list(nombre_col, scores_finales[i], formulas_finales[i], round(cor_max_actual, 4)))
}

# Limpieza de memoria temporal del GA
rm(dt_ga_sample, valores_ga_guardados)
gc(verbose = FALSE)

fwrite(dt_trazabilidad, file = file.path(dir_experimento_base, paste0("GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv")), sep = ",")
cat("\nArchivo de trazabilidad guardado en: GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv\n")
cat("\nColumnas generadas por Algoritmo Genético e inyectadas al dataset:", paste(ga_cols_creadas, collapse = ", "), "\n")

# Blindaje anti-leakage: asegura que clase01 no quede en dataset antes de FEhist
if ("clase01" %in% colnames(dataset)) dataset[, clase01 := NULL]


In [ ]:
# Auditoría de resultados del Algoritmo Genético e inspección de correlaciones
cat("Total individuos en población final:", length(scores_finales), "\n")
cat("Individuos con score > 0.50:", sum(scores_finales > 0.50, na.rm = TRUE), "\n")
cat("Expresiones no triviales con score > 0.50:", sum(scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial), na.rm = TRUE), "\n")
cat("Expresiones únicas no triviales:", length(unique(formulas_finales[scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial)])), "\n")

if (length(ga_cols_creadas) >= 2) {
  cat("\nMatriz de correlación entre las GA_Features seleccionadas (en validate 202107):\n")
  print(round(cor(dataset[foto_mes == 202107, ga_cols_creadas, with = FALSE], use = "pairwise.complete.obs"), 2))
}

cat("\nColumnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):\n")
colnames(dataset)


#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena (blindaje total anti-leakage)
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar","fold_train","fold_final_train"))
)
# Excluye cualquier columna derivada de la clase (ej: clase01_lag, clase01_delta)
campos_buenos <- campos_buenos[!grepl("^clase", campos_buenos)]


In [ ]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

# datos de validation (fijos para todas las semillas)
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= FALSE
)
cat("Filas dvalidate:", nrow(dvalidate), "\n")

# meses de final train
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)
dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# dfinal_train en formato LightGBM (100% de datos, sin undersampling, reutilizable por todas las semillas)
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= FALSE
)
cat("Filas dfinal_train:", nrow(dfinal_train), "\n")

# dataset del futuro para scoring
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
cat("Filas dfuture:", nrow(dfuture), "\n")


In [ ]:
# verificacion de dimensiones
cat("Dimensiones dfinal_train:", nrow(dfinal_train), "x", ncol(dfinal_train), "\n")
cat("Dimensiones dvalidate:", nrow(dvalidate), "x", ncol(dvalidate), "\n")


####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM (plantilla base)
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En x llegan los parametros moviles de LightGBM
# devuelve la AUC en validate del modelo entrenado
# en el parametro x llegan los hiperparametros que se estan optimizando

Estimar_AUC_lightgbm <- function(x, dtrain_local, dval_local, param_fijos_local) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(param_fijos_local, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain_local,
    valids= list(valid = dval_local),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}


seteo del Grid Search

In [ ]:
# Espacio de búsqueda de hiperparámetros (idéntico al baseline)
tb_grid_template <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)


### 9.3.3 Automatización Multi-Semilla y Producción
A continuación se define la función modular que ejecuta el ciclo completo de modelado y producción para cada semilla:
1. Creación de la carpeta `semilla_<semilla>` dentro del directorio del experimento `WF<experimento>`.
2. Undersampling estocástico con la semilla indicada.
3. Grid Search de hiperparámetros (o reutilización según `PARAM$gridsearch_por_semilla`).
4. Final Training del modelo LightGBM con dicha semilla.
5. Exportación de `modelo.txt` e `impo.txt` dentro de la carpeta de la semilla.
6. Scoring de probabilidades en `dfuture` y guardado de `prediccion.txt` dentro de la carpeta de la semilla.
7. Generación y exportación de archivos de corte en `kaggle/` con submit automático.
8. Guardado de metadatos `PARAM.yml` en la carpeta de la semilla.


In [ ]:
# ==============================================================================
# Función modular que ejecuta el ciclo completo para una semilla dada
# ==============================================================================
ejecutar_experimento_semilla <- function(vsemilla, mejores_hiperparametros_compartidos = NULL) {

  cat("\n===================================================================\n")
  cat(">>> INICIANDO EJECUCION PARA SEMILLA:", vsemilla, "<<<\n")
  cat("===================================================================\n")

  # 1. Crear carpeta identificatoria de la semilla dentro del experimento
  dir_semilla <- file.path(dir_experimento_base, paste0("semilla_", vsemilla))
  dir.create(dir_semilla, showWarnings = FALSE, recursive = TRUE)
  dir_kaggle <- file.path(dir_semilla, "kaggle")
  dir.create(dir_kaggle, showWarnings = FALSE, recursive = TRUE)

  # Parámetros para esta semilla
  PARAM_sem <- copy(PARAM)
  PARAM_sem$semilla_actual <- vsemilla
  PARAM_sem$lgbm$param_fijos$seed <- vsemilla

  # 2. Undersampling con la semilla actual
  set.seed(vsemilla, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]
  dataset[, fold_train := foto_mes %in% PARAM_sem$trainingstrategy$training & 
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") | 
     azar < PARAM_sem$trainingstrategy$training_pct ) ]

  dtrain_sem <- lgb.Dataset(
    data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_train == TRUE, clase01],
    free_raw_data = TRUE
  )

  # 3. Grid Search de Hiperparámetros
  arch_grid <- file.path(dir_semilla, "tb_grid_search_01.txt")
  mejores_hiper <- NULL
  auc_optima <- NULL

  if( file.exists(arch_grid) ) {
    cat("Cargando grid search previo de:", arch_grid, "\n")
    tb_nueva <- fread(arch_grid)
    setorder( tb_nueva, -AUC )
    auc_optima <- tb_nueva[1, AUC]
    mejores_hiper <- as.list(tb_nueva[1])
    mejores_hiper$AUC <- NULL
  } else if( is.null(mejores_hiperparametros_compartidos) || isTRUE(PARAM$gridsearch_por_semilla) ) {
    cat("Ejecutando Grid Search para semilla:", vsemilla, "...\n")
    tb_nueva <- copy(tb_grid_template)

    tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(
      .SD,
      dtrain_local = dtrain_sem,
      dval_local = dvalidate,
      param_fijos_local = PARAM_sem$lgbm$param_fijos
    ), by = 1:nrow(tb_nueva) ]

    fwrite( tb_nueva, file = arch_grid, sep = "\t" )
    setorder( tb_nueva, -AUC )
    auc_optima <- tb_nueva[1, AUC]
    mejores_hiper <- as.list(tb_nueva[1])
    mejores_hiper$AUC <- NULL
  } else {
    cat("Reutilizando mejores hiperparámetros compartidos para semilla:", vsemilla, "...\n")
    mejores_hiper <- copy(mejores_hiperparametros_compartidos)
    auc_optima <- PARAM$out$lgbm$AUC
  }

  PARAM_sem$out$lgbm$AUC <- auc_optima
  PARAM_sem$out$lgbm$mejores_hiperparametros <- mejores_hiper

  # 4. Final Training
  cat("Entrenando modelo final para semilla:", vsemilla, "...\n")
  fijos <- copy(PARAM_sem$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL

  param_final <- c(fijos, mejores_hiper)
  param_final$seed <- vsemilla

  set.seed(vsemilla, kind = "L'Ecuyer-CMRG")
  final_model <- lgb.train(
    data = dfinal_train,
    param = param_final,
    verbose = -100
  )

  # Grabo modelo en la carpeta de la semilla
  arch_modelo <- file.path(dir_semilla, "modelo.txt")
  lgb.save(final_model, arch_modelo)

  # Importancia de variables en la carpeta de la semilla
  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite( tb_importancia, file = file.path(dir_semilla, "impo.txt"), sep = "\t" )

  # 5. Scoring sobre dfuture
  cat("Generando predicciones sobre dfuture para semilla:", vsemilla, "...\n")
  prediccion <- predict(
    final_model,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  fwrite( tb_prediccion, file = file.path(dir_semilla, "prediccion.txt"), sep = "\t" )

  # 6. Kaggle Competition Submit
  PARAM_sem$kaggle$competencia <- "utn-2026-virtual-jr"
  PARAM_sem$kaggle$cortes <- seq(1800, 2400, by = 100)

  setorder(tb_prediccion, -prob)

  for (envios in PARAM_sem$kaggle$cortes) {
    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]

    archivo_kaggle <- file.path(dir_kaggle, paste0("KA", PARAM_sem$experimento, "_s", vsemilla, "_", envios, ".csv"))

    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ","
    )

    comando <- "kaggle competitions submit"
    competencia <- paste("-c", PARAM_sem$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)
    mensaje <- paste0("-m 'envios=", envios, "  semilla=", vsemilla, "'")
    linea <- paste(comando, competencia, arch, mensaje)

    tryCatch({
      salida <- system(linea, intern = TRUE)
      cat(salida, "\n")
      Sys.sleep(5)
    }, error = function(e) {
      cat("Aviso submit Kaggle:", conditionMessage(e), "\n")
    })
  }

  # 7. Grabo los parametros en la carpeta de la semilla
  if( !require("yaml")) install.packages("yaml")
  require("yaml")
  write_yaml( PARAM_sem, file = file.path(dir_semilla, "PARAM.yml") )

  # Limpieza de memoria
  rm(dtrain_sem, final_model)
  gc(full = TRUE, verbose = FALSE)

  cat(">>> FINALIZADA EXITOSAMENTE SEMILLA:", vsemilla, "<<<\n\n")

  return( list(mejores_hiper = mejores_hiper, AUC = auc_optima) )
}


##### Bucle de Automatización Multi-Semilla


In [ ]:
# ==============================================================================
# Bucle de automatización para todas las semillas configuradas
# ==============================================================================
cat("Semillas a ejecutar:", paste(PARAM$semillas, collapse = ", "), "\n")

mejores_hiper_compartidos <- NULL

for( idx in seq_along(PARAM$semillas) ) {
  sem <- PARAM$semillas[idx]

  res <- ejecutar_experimento_semilla(
    vsemilla = sem,
    mejores_hiperparametros_compartidos = mejores_hiper_compartidos
  )

  # Si se optimiza una sola vez, guardamos los mejores hiperparámetros para las siguientes semillas
  if( idx == 1 && !isTRUE(PARAM$gridsearch_por_semilla) ) {
    mejores_hiper_compartidos <- res$mejores_hiper
    PARAM$out$lgbm$AUC <- res$AUC
    PARAM$out$lgbm$mejores_hiperparametros <- res$mejores_hiper
  }
}


##### Ensamble Multimodelo (Blending)


In [ ]:
# ==============================================================================
# Ensamble Multimodelo (Blending) sobre todas las semillas ejecutadas
# ==============================================================================
if( length(PARAM$semillas) > 1 ) {
  cat("\nConstruyendo ensamble de probabilidades para", length(PARAM$semillas), "semillas...\n")

  tb_ensemble <- dfuture[, list(numero_de_cliente)]
  tb_ensemble[, prob_total := 0]
  semillas_validas <- 0

  for( sem in PARAM$semillas ) {
    arch_pred <- file.path(dir_experimento_base, paste0("semilla_", sem), "prediccion.txt")
    if( file.exists(arch_pred) ) {
      tb_sem <- fread(arch_pred)
      tb_ensemble[, paste0("prob_s", sem) := tb_sem$prob]
      tb_ensemble[, prob_total := prob_total + tb_sem$prob]
      semillas_validas <- semillas_validas + 1
    }
  }

  if( semillas_validas > 1 ) {
    tb_ensemble[, prob := prob_total / semillas_validas]
    tb_ensemble[, prob_total := NULL]

    fwrite( tb_ensemble,
      file = file.path(dir_experimento_base, "prediccion_ensemble.txt"),
      sep = "\t"
    )

    dir_kaggle_ens <- file.path(dir_experimento_base, "kaggle_ensemble")
    dir.create(dir_kaggle_ens, showWarnings = FALSE)

    setorder( tb_ensemble, -prob )

    for (envios in PARAM$kaggle$cortes) {
      tb_ensemble[, Predicted := 0L]
      tb_ensemble[1:envios, Predicted := 1L]

      archivo_kaggle_ens <- file.path(dir_kaggle_ens, paste0("KA", PARAM$experimento, "_ensemble_", semillas_validas, "sem_", envios, ".csv"))

      fwrite( tb_ensemble[, list(numero_de_cliente, Predicted)],
        file = archivo_kaggle_ens,
        sep = ","
      )
    }
    cat("Ensamble generado exitosamente en:", dir_kaggle_ens, "\n")
  }
}


### 9.3.4 Finalización del Workflow


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
